# 00 · Coverage (Step 0)

Pick the study area, LiDAR acquisition and CRS before any processing (brief, Phase 0).

**Outputs**
- `figures/00_lidar_tnm_projects.csv` — 3DEP LPC projects from The National Map
- `figures/00_lidar_pc_projects.csv` — 3DEP COPC projects on Planetary Computer, with point density
- `figures/00_gedi_granules_by_year.csv` — GEDI L2A granules over the AOI, by year and leaf-on
- `figures/00_coverage_map.png`

**Runtime:** Colab → *Runtime › Change runtime type › R*. First run installs packages (~3–5 min).

In [ ]:
# --- Setup -------------------------------------------------------------------
REPO_URL  <- "https://github.com/MegaDeadCowboy/forest-height-s2-lidar.git"
REPO_NAME <- "forest-height-s2-lidar"
if (dir.exists("/content") && !dir.exists(file.path("/content", REPO_NAME)))
  system(paste("git clone --depth 1", REPO_URL, file.path("/content", REPO_NAME)))
if (dir.exists("/content")) setwd(file.path("/content", REPO_NAME)) else if (basename(getwd()) == "notebooks") setwd("..")

source("R/setup.R"); source("R/config.R"); source("R/aoi.R"); source("R/coverage.R")
setup_packages(c("sf", "dplyr", "tidyr", "ggplot2", "httr2", "jsonlite", "rstac", "purrr", "stringr", "readr"))

DATA <- "data"   # single data root; 00 outputs are small. Big rasters go to Drive from 01 on.
dir.create(DATA, showWarnings = FALSE); dir.create("figures", showWarnings = FALSE)

## 1 · Study area
HUC12 watersheds intersecting the Lake Whatcom search box. Check the printed names, then edit `AOI_PATTERN` if the selection is wrong (regex on `name`).

In [ ]:
AOI_PATTERN <- "Whatcom"
a <- load_aoi(AOI_BBOX, pattern = AOI_PATTERN)
if (!is.null(a$huc12)) print(sf::st_drop_geometry(a$huc12)[, c("huc12", "name", "areasqkm")])
cat(sprintf("\nAOI area: %.1f km²\n", as.numeric(sf::st_area(a$aoi)) / 1e6))

## 2 · 3DEP LiDAR — The National Map (authoritative tile list + AOI coverage)

In [ ]:
tnm <- tryCatch(tnm_lpc_products(AOI_BBOX), error = function(e) { message("TNM failed: ", conditionMessage(e)); tibble::tibble() })
tnm_sf <- tnm_tiles_sf(tnm)
tnm_in <- tnm_sf[sf::st_intersects(tnm_sf, a$aoi, sparse = FALSE)[, 1], ]

tnm_proj <- tnm_in |>
  sf::st_drop_geometry() |>
  dplyr::group_by(project, workunit, year) |>
  dplyr::summarise(n_tiles = dplyr::n(),
                   size_gb = round(sum(as.numeric(sizeInBytes), na.rm = TRUE) / 1e9, 1),
                   published = max(publicationDate, na.rm = TRUE), .groups = "drop") |>
  dplyr::arrange(dplyr::desc(year))
tnm_proj$aoi_cover_pct <- sapply(tnm_proj$workunit, function(w) {
  fp <- sf::st_union(tnm_in[tnm_in$workunit == w, ])
  round(100 * as.numeric(sf::st_area(sf::st_intersection(fp, a$aoi))) / as.numeric(sf::st_area(a$aoi)), 1)
})
readr::write_csv(tnm_proj, "figures/00_lidar_tnm_projects.csv")
print(tnm_proj, n = 50, width = Inf)

## 3 · 3DEP COPC on Planetary Computer (metadata only)
Flight dates and rough density for tiles in the search box. Density is all returns per m² of tile footprint (rough); exact density and per-tile dates come from LAS headers in 01.

In [ ]:
pc <- tryCatch(pc_copc_items(AOI_BBOX), error = function(e) { message("PC failed: ", conditionMessage(e)); NULL })

if (!is.null(pc)) {
  pc_proj <- pc |>
    sf::st_drop_geometry() |>
    dplyr::group_by(project, year) |>
    dplyr::summarise(n_tiles = dplyr::n(),
                     flight_start = min(start, na.rm = TRUE), flight_end = max(end, na.rm = TRUE),
                     pts_m2_p10 = round(quantile(pts_m2, 0.1, na.rm = TRUE), 1),
                     pts_m2_median = round(median(pts_m2, na.rm = TRUE), 1), .groups = "drop")
  readr::write_csv(pc_proj, "figures/00_lidar_pc_projects.csv")
  print(pc_proj, n = 50, width = Inf)
}

## 4 · WA DNR Lidar Portal (manual check)
The DNR portal has no query API. Open https://lidarportal.dnr.wa.gov, zoom to Lake Whatcom, turn on the project boundaries, and record what covers the AOI below. Fill in what the portal metadata says — leave `NA` rather than guessing.

In [ ]:
# From DNR portal metadata (WGS QA report + NV5 technical report). Unknowns stay NA.
dnr <- tibble::tribble(
  ~project,                   ~flown,                               ~vendor, ~first_ret_pts_m2, ~ground_pts_m2, ~crs,                                    ~format,         ~notes,
  "Cascades North 2023",      NA,                                   NA,      NA,                NA,             NA,                                      NA,              "covers west AOI + main lake; metadata pending",
  "Acme Valley Refresh23 2024","2023-07-28; 2023-09-22; 2024-02-04", "NV5",   11.0,              4.9,            "WA SP South, NAD83(HARN), NAVD88 G18, US ft", "LAS 1.4 (.las)", "fills Smith Ck / south basin; low elev flown leaf-off (Feb 2024); classes 1,2,7,9,17,18"
)
readr::write_csv(dnr, "figures/00_lidar_dnr_projects.csv")
dnr

## 5 · GEDI L2A granules (NASA CMR, no login)
Counts **orbits** crossing the search box, not quality shots inside the LiDAR footprint. Shot counts need the HDF5 subset, done once the acquisition is picked (start of 02). Leaf-on = Jun–Sep.

In [ ]:
gedi <- tryCatch(cmr_gedi_granules(AOI_BBOX), error = function(e) { message("CMR failed: ", conditionMessage(e)); tibble::tibble() })

gedi_yr <- gedi |>
  dplyr::count(version, year, leaf_on) |>
  tidyr::pivot_wider(names_from = leaf_on, values_from = n, values_fill = 0,
                     names_prefix = "leaf_on_") |>
  dplyr::arrange(version, year)
readr::write_csv(gedi_yr, "figures/00_gedi_granules_by_year.csv")
print(gedi_yr, n = 50)

## 6 · Coverage map

In [ ]:
p <- ggplot2::ggplot()
if (!is.null(a$huc12)) p <- p + ggplot2::geom_sf(data = sf::st_transform(a$huc12, CRS_EPSG), fill = NA, colour = "grey70", linewidth = 0.2)
if (nrow(tnm_in)) p <- p + ggplot2::geom_sf(data = tnm_in, ggplot2::aes(fill = project), colour = "white", linewidth = 0.1, alpha = 0.5)
p <- p +
  ggplot2::geom_sf(data = a$aoi, fill = NA, colour = "black", linewidth = 0.6) +
  ggplot2::coord_sf(crs = CRS_EPSG) +
  ggplot2::labs(title = "3DEP LiDAR coverage, Lake Whatcom AOI", fill = "3DEP project",
                caption = sprintf("USGS TNM LPC tile extents · WBD HUC12 · EPSG:%d", CRS_EPSG)) +
  ggplot2::theme_minimal(base_size = 10) + ggplot2::theme(legend.position = "bottom", legend.direction = "vertical")
ggplot2::ggsave("figures/00_coverage_map.png", p, width = 7, height = 7, dpi = 200, bg = "white")
p

## 7 · Decision
Paste back to Claude: the HUC12 table (cell 1), the two project tables (cells 2–3), the DNR rows (cell 4) and the GEDI table (cell 5).

Then decide, against the brief's Step 0 criteria:
- [ ] Acquisition: flown **2019 or later**, covers most of the AOI, density recorded (*verify* until read from LAS headers in 01)
- [ ] GEDI leaf-on granules in years near the flight
- [ ] CRS: native LiDAR CRS noted; working CRS confirmed (default EPSG:32610)
- [ ] Test tile (≤1 km², forested, with a stream) picked for 01
- [ ] Decision log lines added to `docs/project_brief.md`